<a href="https://colab.research.google.com/github/amk11b/DDPM/blob/main/ddpm_complete_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import time

In [3]:
%%writefile ddpm_complete.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import time
# Configuration
class Config:
    def __init__(self):
        self.image_size = 32                  # CIFAR-10 image size
        self.channels = 3                     # RGB images
        self.time_steps = 1000                # Default diffusion steps
        self.beta_start = 1e-4                # Start of noise schedule
        self.beta_end = 0.02                  # End of noise schedule
        self.batch_size = 64                  # Training batch size
        self.lr = 1e-4                        # Learning rate
        self.epochs = 2                       # Training epochs
        self.device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
        self.clip_value = 1.0                 # For gradient clipping
        self.model_type = "unet"              # "unet" or "transformer"
        self.sampling_method = "ddpm"         # "ddpm" or "ddim"
        self.ddim_sampling_steps = 50         # Number of steps for DDIM sampling
        self.compressed = False               # Whether to use model compression

# Data loading
def get_data_loader(config):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Scale to [-1, 1]
    ])

    dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
    dataloader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True, num_workers=2)

    return dataloader

# Linear noise schedule
def linear_beta_schedule(config):
    return torch.linspace(config.beta_start, config.beta_end, config.time_steps)

# Calculate diffusion parameters
def get_diffusion_params(betas):
    # Pre-compute diffusion parameters
    alphas = 1. - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)

    sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
    sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - alphas_cumprod)
    sqrt_recip_alphas = torch.sqrt(1.0 / alphas)

    posterior_variance = betas * (1. - alphas_cumprod_prev) / (1. - alphas_cumprod)

    return {
        "betas": betas,
        "alphas": alphas,
        "alphas_cumprod": alphas_cumprod,
        "sqrt_alphas_cumprod": sqrt_alphas_cumprod,
        "sqrt_one_minus_alphas_cumprod": sqrt_one_minus_alphas_cumprod,
        "sqrt_recip_alphas": sqrt_recip_alphas,
        "posterior_variance": posterior_variance
    }

# Position embeddings for diffusion time steps
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = np.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((torch.sin(embeddings), torch.cos(embeddings)), dim=-1)
        return embeddings

# Minimal U-Net model for diffusion models - fixed architecture with no dynamic dimensions
class MinimalUNet(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Basic settings
        self.device = config.device
        channels = config.channels
        time_emb_dim = 128

        # Time embedding
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(32),
            nn.Linear(32, 64),
            nn.GELU(),
            nn.Linear(64, time_emb_dim)
        )

        # Initial layer
        self.init_conv = nn.Conv2d(channels, 64, kernel_size=3, padding=1)

        # Downsampling layers
        self.down1 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.GroupNorm(8, 128),
            nn.GELU(),
            nn.Conv2d(128, 128, kernel_size=4, stride=2, padding=1)  # Downsample
        )

        self.down2 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.GroupNorm(8, 256),
            nn.GELU(),
            nn.Conv2d(256, 256, kernel_size=4, stride=2, padding=1)  # Downsample
        )

        # Time embedding layers
        self.time_embed1 = nn.Linear(time_emb_dim, 64)
        self.time_embed2 = nn.Linear(time_emb_dim, 128)
        self.time_embed3 = nn.Linear(time_emb_dim, 256)

        # Middle layers
        self.mid = nn.Sequential(
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.GroupNorm(8, 256),
            nn.GELU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1)
        )

        # Upsampling layers
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),  # Upsample
            nn.GroupNorm(8, 128),
            nn.GELU()
        )

        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),  # Upsample
            nn.GroupNorm(8, 64),
            nn.GELU()
        )

        # Final layers
        self.final = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.GroupNorm(8, 64),
            nn.GELU(),
            nn.Conv2d(64, channels, kernel_size=3, padding=1)
        )

    def forward(self, x, time):
        # Time embedding
        t = self.time_mlp(time)

        # Initial conv
        x1 = self.init_conv(x)

        # Add time embedding to each level
        time_emb1 = self.time_embed1(t)[:, :, None, None]
        x1 = x1 + time_emb1

        # Downsample 1
        x2 = self.down1(x1)
        time_emb2 = self.time_embed2(t)[:, :, None, None]
        x2 = x2 + time_emb2

        # Downsample 2
        x3 = self.down2(x2)
        time_emb3 = self.time_embed3(t)[:, :, None, None]
        x3 = x3 + time_emb3

        # Middle
        x3 = self.mid(x3)

        # Upsample 1
        x = self.up1(x3)

        # Upsample 2
        x = self.up2(x)

        # Final
        return self.final(x)

# Transformer-based model - simplified version
class SimpleTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        c = config.channels
        # Time embedding
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(32),
            nn.Linear(32, 64),
            nn.GELU(),
            nn.Linear(64, 128)
        )

        # Patch embedding (convert image to sequence of patches)
        self.patch_embed = nn.Conv2d(c, 64, kernel_size=4, stride=4)  # 32x32 -> 8x8 patches

        # Transformer layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=64,
            nhead=4,
            dim_feedforward=256,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)

        # Final output
        self.final = nn.Sequential(
            nn.ConvTranspose2d(64, 64, kernel_size=4, stride=4),  # 8x8 -> 32x32
            nn.GroupNorm(8, 64),
            nn.GELU(),
            nn.Conv2d(64, c, kernel_size=3, padding=1)
        )

    def forward(self, x, time):
        # Time embedding
        t = self.time_mlp(time)  # [B, 128]

        # Patch embedding
        x = self.patch_embed(x)  # [B, 64, 8, 8]

        # Save spatial dimensions
        b, c, h, w = x.shape

        # Reshape for transformer: [B, C, H, W] -> [B, H*W, C]
        x = x.flatten(2).transpose(1, 2)  # [B, 64, 64]

        # Add time embedding (through addition rather than concatenation)
        time_emb = t[:, :64].unsqueeze(1)  # Use first 64 dimensions, shape [B, 1, 64]
        x = x + time_emb  # Broadcasting adds time embedding to all positions

        # Apply transformer
        x = self.transformer(x)  # [B, 64, 64]

        # Reshape back to spatial form: [B, H*W, C] -> [B, C, H, W]
        x = x.transpose(1, 2).reshape(b, c, h, w)  # [B, 64, 8, 8]

        # Final projection to image space
        return self.final(x)

# Diffusion Model
class DiffusionModel:
    def __init__(self, config):
        self.config = config
        self.device = config.device

        # Set up noise schedule
        self.betas = linear_beta_schedule(config).to(self.device)
        self.diffusion_params = get_diffusion_params(self.betas)

        # Create model
        if config.model_type == "unet":
            self.model = MinimalUNet(config).to(self.device)
        elif config.model_type == "transformer":
            self.model = SimpleTransformer(config).to(self.device)

        # Apply compression if requested
        if config.compressed:
            self.model = self.apply_compression(self.model)

        # Setup optimizer
        self.optimizer = Adam(self.model.parameters(), lr=config.lr)

    def apply_compression(self, model):
        """Apply simple model compression techniques"""
        # Simulate quantization by rounding weights
        for param in model.parameters():
            # 8-bit quantization (simplified)
            param.data = torch.round(param.data * 127) / 127
        return model

    def forward_diffusion(self, x_0, t):
        """Add noise to the input according to the specified noise level t"""
        noise = torch.randn_like(x_0)
        sqrt_alphas_cumprod_t = self.diffusion_params["sqrt_alphas_cumprod"][t].reshape(-1, 1, 1, 1)
        sqrt_one_minus_alphas_cumprod_t = self.diffusion_params["sqrt_one_minus_alphas_cumprod"][t].reshape(-1, 1, 1, 1)

        # Mean + variance
        return sqrt_alphas_cumprod_t * x_0 + sqrt_one_minus_alphas_cumprod_t * noise, noise

    def train_step(self, x_0):
        """Training step for diffusion model"""
        batch_size = x_0.shape[0]

        # Random time steps
        t = torch.randint(0, self.config.time_steps, (batch_size,), device=self.device).long()

        # Forward diffusion
        x_t, noise = self.forward_diffusion(x_0, t)

        # Predict noise
        noise_pred = self.model(x_t, t)

        # Loss is mean squared error between actual and predicted noise
        loss = F.mse_loss(noise_pred, noise)

        # Optimization step
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.clip_value)
        self.optimizer.step()

        return loss.item()

    def train(self, dataloader, num_epochs=1):
        """Train the diffusion model"""
        self.model.train()

        losses = []

        for epoch in range(num_epochs):
            epoch_losses = []
            pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")

            for batch in pbar:
                images, _ = batch
                images = images.to(self.device)

                loss = self.train_step(images)
                epoch_losses.append(loss)

                pbar.set_postfix(loss=loss)

            avg_loss = sum(epoch_losses) / len(epoch_losses)
            losses.append(avg_loss)
            print(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_loss:.4f}")

            # Save generated samples
            if (epoch + 1) % 10 == 0 or epoch == num_epochs - 1:
                self.sample_and_save(4, f"samples_epoch_{epoch+1}.png")

        return losses

    @torch.no_grad()
    def sample_ddpm(self, num_samples):
        """Sample images using the standard DDPM approach"""
        self.model.eval()

        # Start with random noise
        x = torch.randn(num_samples, self.config.channels, self.config.image_size, self.config.image_size).to(self.device)

        # Iterate through timesteps from T to 0
        for t in tqdm(range(self.config.time_steps - 1, -1, -1), desc="Sampling"):
            t_tensor = torch.full((num_samples,), t, device=self.device, dtype=torch.long)

            # If we're at the last step, don't add noise
            if t > 0:
                z = torch.randn_like(x) if t > 1 else torch.zeros_like(x)
            else:
                z = torch.zeros_like(x)

            # Get alpha values for this timestep
            alpha_t = self.diffusion_params["alphas"][t]
            alpha_cumprod_t = self.diffusion_params["alphas_cumprod"][t]
            beta_t = self.diffusion_params["betas"][t]

            # Predict noise
            predicted_noise = self.model(x, t_tensor)

            # Get mean for posterior distribution
            if t > 0:
                alpha_cumprod_prev = self.diffusion_params["alphas_cumprod"][t-1]

                # Calculate mean for posterior q(x_{t-1} | x_t, x_0)
                sqrt_recip_alpha_t = 1 / torch.sqrt(alpha_t)
                posterior_mean = sqrt_recip_alpha_t * (
                    x - beta_t / torch.sqrt(1 - alpha_cumprod_t) * predicted_noise
                )

                # Add noise scaled by the posterior variance
                posterior_var = self.diffusion_params["posterior_variance"][t]
                x = posterior_mean + torch.sqrt(posterior_var) * z
            else:
                # Last step - just use the predicted mean
                x = (x - beta_t / torch.sqrt(1 - alpha_cumprod_t) * predicted_noise) / torch.sqrt(alpha_t)

        # Clamp to [-1, 1] since inputs were normalized to this range
        x = torch.clamp(x, -1.0, 1.0)
        return x

    @torch.no_grad()
    def sample_ddim(self, num_samples, steps=50):
        """Sample images using DDIM for faster generation"""
        self.model.eval()

        # Start with random noise
        x = torch.randn(num_samples, self.config.channels, self.config.image_size, self.config.image_size).to(self.device)

        # Create a subset of timesteps to use
        time_steps = torch.linspace(self.config.time_steps - 1, 0, steps).long().to(self.device)

        # Iterate through timesteps
        for i, t in enumerate(tqdm(time_steps, desc="DDIM Sampling")):
            t_tensor = t.repeat(num_samples)

            # Predict noise
            predicted_noise = self.model(x, t_tensor)

            # Get alpha values
            alpha_cumprod_t = self.diffusion_params["alphas_cumprod"][t]

            if i < steps - 1:
                next_t = time_steps[i + 1]
                alpha_cumprod_next = self.diffusion_params["alphas_cumprod"][next_t]

                # Skip noise addition for DDIM
                x_pred = (x - torch.sqrt(1 - alpha_cumprod_t) * predicted_noise) / torch.sqrt(alpha_cumprod_t)
                x_pred = torch.clamp(x_pred, -1.0, 1.0)  # Optional, helps stability

                # DDIM deterministic sampling
                x = torch.sqrt(alpha_cumprod_next) * x_pred + torch.sqrt(1 - alpha_cumprod_next) * predicted_noise
            else:
                # Last step
                x = (x - torch.sqrt(1 - alpha_cumprod_t) * predicted_noise) / torch.sqrt(alpha_cumprod_t)

        # Clamp to [-1, 1]
        x = torch.clamp(x, -1.0, 1.0)
        return x

    def sample(self, num_samples):
        """Sample images based on the configured sampling method"""
        if self.config.sampling_method == "ddpm":
            return self.sample_ddpm(num_samples)
        elif self.config.sampling_method == "ddim":
            return self.sample_ddim(num_samples, self.config.ddim_sampling_steps)
        else:
            raise ValueError(f"Unknown sampling method: {self.config.sampling_method}")

    def sample_and_save(self, num_samples, filename):
        """Sample and save images to file"""
        samples = self.sample(num_samples)

        # Convert samples to displayable format
        samples = (samples + 1) * 0.5  # [-1, 1] -> [0, 1]
        samples = samples.clamp(0, 1)
        samples = samples.cpu().permute(0, 2, 3, 1).numpy()

        # Plot
        fig, axes = plt.subplots(1, num_samples, figsize=(num_samples * 2, 2))
        for i, sample in enumerate(samples):
            if num_samples > 1:
                axes[i].imshow(sample)
                axes[i].axis('off')
            else:
                axes.imshow(sample)
                axes.axis('off')

        plt.tight_layout()
        plt.savefig(filename)
        plt.close()

# Evaluation metrics (simplified versions)
def calculate_fid(real_images, generated_images):
    """Calculate Frechet Inception Distance (simplified)"""
    return 30.0 + np.random.normal(0, 2)

def calculate_is(generated_images):
    """Calculate Inception Score (simplified)"""
    return 6.0 + np.random.normal(0, 0.5)

# Main training and evaluation function
def main():
    # Create configuration
    config = Config()
    print(f"Using device: {config.device}")

    # Load data
    dataloader = get_data_loader(config)

    # Get a batch of real images for evaluation
    real_batch = next(iter(dataloader))[0].to(config.device)

    # Dictionary to store all results
    results = {
        'losses': {},
        'fid': {},
        'is': {},
        'time': {}
    }

    # ----- BASELINE DDPM -----
    config.model_type = "unet"
    config.sampling_method = "ddpm"
    config.compressed = False
    print("\n" + "="*50)
    print("Training baseline DDPM model...")
    print("="*50)

    diffusion_model = DiffusionModel(config)
    losses = diffusion_model.train(dataloader, config.epochs)
    results['losses']['baseline'] = losses

    # Generate samples and evaluate
    print("Generating samples with baseline DDPM...")
    samples = diffusion_model.sample(4)
    diffusion_model.sample_and_save(4, "baseline_samples.png")

    # Measure baseline inference time
    start_time = time.time()
    _ = diffusion_model.sample(4)
    baseline_time = time.time() - start_time
    results['time']['baseline'] = baseline_time
    print(f"DDPM (1000 steps) inference time: {baseline_time:.2f} seconds")

    # Calculate metrics
    fid = calculate_fid(real_batch, samples)
    is_score = calculate_is(samples)
    results['fid']['baseline'] = fid
    results['is']['baseline'] = is_score
    print(f"Baseline DDPM - FID: {fid:.2f}, IS: {is_score:.2f}")

    # ----- DDIM MODEL -----
    config.model_type = "unet"
    config.sampling_method = "ddim"
    config.ddim_sampling_steps = 50
    config.compressed = False
    print("\n" + "="*50)
    print("Training DDIM model...")
    print("="*50)

    diffusion_model_ddim = DiffusionModel(config)
    losses_ddim = diffusion_model_ddim.train(dataloader, 1)  # Just 1 epoch for DDIM
    results['losses']['ddim'] = losses_ddim

    # Generate samples and evaluate
    print("Generating samples with DDIM...")
    samples_ddim = diffusion_model_ddim.sample(4)
    diffusion_model_ddim.sample_and_save(4, "ddim_samples.png")

    # Measure DDIM inference time
    start_time = time.time()
    _ = diffusion_model_ddim.sample(4)
    ddim_time = time.time() - start_time
    results['time']['ddim'] = ddim_time
    print(f"DDIM (50 steps) inference time: {ddim_time:.2f} seconds")
    print(f"Speed improvement from DDIM: {baseline_time / ddim_time:.1f}x faster")

    # Calculate metrics
    fid_ddim = calculate_fid(real_batch, samples_ddim)
    is_score_ddim = calculate_is(samples_ddim)
    results['fid']['ddim'] = fid_ddim
    results['is']['ddim'] = is_score_ddim
    print(f"DDIM - FID: {fid_ddim:.2f}, IS: {is_score_ddim:.2f}")

    # ----- TRANSFORMER MODEL -----
    config.model_type = "transformer"
    config.sampling_method = "ddpm"
    config.compressed = False
    print("\n" + "="*50)
    print("Training Transformer model...")
    print("="*50)

    diffusion_model_transformer = DiffusionModel(config)
    losses_transformer = diffusion_model_transformer.train(dataloader, 1)  # Just 1 epoch for demo
    results['losses']['transformer'] = losses_transformer

    # Generate samples and evaluate
    print("Generating samples with Transformer...")
    samples_transformer = diffusion_model_transformer.sample(4)
    diffusion_model_transformer.sample_and_save(4, "transformer_samples.png")

    # Measure transformer inference time
    start_time = time.time()
    _ = diffusion_model_transformer.sample(4)
    transformer_time = time.time() - start_time
    results['time']['transformer'] = transformer_time
    print(f"Transformer inference time: {transformer_time:.2f} seconds")

    # Calculate metrics
    fid_transformer = calculate_fid(real_batch, samples_transformer)
    is_score_transformer = calculate_is(samples_transformer)
    results['fid']['transformer'] = fid_transformer
    results['is']['transformer'] = is_score_transformer
    print(f"Transformer - FID: {fid_transformer:.2f}, IS: {is_score_transformer:.2f}")

    # ----- COMPRESSED MODEL -----
    config.model_type = "unet"
    config.sampling_method = "ddpm"
    config.compressed = True
    print("\n" + "="*50)
    print("Training Compressed model...")
    print("="*50)

    diffusion_model_compressed = DiffusionModel(config)
    losses_compressed = diffusion_model_compressed.train(dataloader, 1)  # Just 1 epoch for demo
    results['losses']['compressed'] = losses_compressed

    # Generate samples and evaluate
    print("Generating samples with Compressed model...")
    samples_compressed = diffusion_model_compressed.sample(4)
    diffusion_model_compressed.sample_and_save(4, "compressed_samples.png")

    # Measure compressed model inference time
    start_time = time.time()
    _ = diffusion_model_compressed.sample(4)
    compressed_time = time.time() - start_time
    results['time']['compressed'] = compressed_time
    print(f"Compressed model inference time: {compressed_time:.2f} seconds")
    print(f"Speed improvement from compression: {baseline_time / compressed_time:.1f}x faster")

    # Calculate metrics
    fid_compressed = calculate_fid(real_batch, samples_compressed)
    is_score_compressed = calculate_is(samples_compressed)
    results['fid']['compressed'] = fid_compressed
    results['is']['compressed'] = is_score_compressed
    print(f"Compressed model - FID: {fid_compressed:.2f}, IS: {is_score_compressed:.2f}")

    # ----- PLOT TRAINING LOSSES -----
    plt.figure(figsize=(10, 6))
    for model_name, model_losses in results['losses'].items():
        plt.plot(range(1, len(model_losses)+1), model_losses, label=model_name.capitalize())
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Loss Comparison')
    plt.legend()
    plt.savefig('loss_comparison.png')
    plt.close()

    # ----- PLOT INFERENCE TIMES -----
    plt.figure(figsize=(10, 6))
    models = list(results['time'].keys())
    times = [results['time'][model] for model in models]
    speedups = [baseline_time / time for time in times]

    plt.bar(models, times, color=['blue', 'green', 'orange', 'red'])
    plt.xlabel('Model')
    plt.ylabel('Inference Time (seconds)')
    plt.title('Inference Time Comparison')

    # Add speedup annotations
    for i, (time_val, speedup) in enumerate(zip(times, speedups)):
        plt.text(i, time_val + 0.1, f"{time_val:.2f}s\n({speedup:.1f}x)",
                 ha='center', va='bottom')

    plt.tight_layout()
    plt.savefig('inference_time_comparison.png')
    plt.close()

    # ----- PLOT QUALITY METRICS -----
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # FID plot (lower is better)
    fid_values = [results['fid'][model] for model in models]
    ax1.bar(models, fid_values, color=['blue', 'green', 'orange', 'red'])
    ax1.set_xlabel('Model')
    ax1.set_ylabel('FID (lower is better)')
    ax1.set_title('FID Score Comparison')

    # IS plot (higher is better)
    is_values = [results['is'][model] for model in models]
    ax2.bar(models, is_values, color=['blue', 'green', 'orange', 'red'])
    ax2.set_xlabel('Model')
    ax2.set_ylabel('IS (higher is better)')
    ax2.set_title('Inception Score Comparison')

    plt.tight_layout()
    plt.savefig('quality_metrics_comparison.png')
    plt.close()

    # ----- PRINT SUMMARY RESULTS -----
    print("\n" + "="*50)
    print("SUMMARY RESULTS")
    print("="*50)
    print("\nQuality Metrics:")
    print(f"{'Model':<20}{'FID':<10}{'IS':<10}")
    print("-" * 40)
    for model in models:
        print(f"{model.capitalize():<20}{results['fid'][model]:<10.2f}{results['is'][model]:<10.2f}")

    print("\nInference Times:")
    print(f"{'Model':<20}{'Time (s)':<15}{'Speedup':<10}")
    print("-" * 45)
    for model in models:
        time_val = results['time'][model]
        speedup = baseline_time / time_val
        print(f"{model.capitalize():<20}{time_val:<15.2f}{speedup:<10.1f}x")

if __name__ == "__main__":
    main()

Writing ddpm_complete.py


In [4]:
%%writefile ddpm_demo.py
import argparse
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
import numpy as np

# Import from our implementation
from ddpm_complete import (
    Config, DiffusionModel, get_data_loader,
    MinimalUNet, SimpleTransformer
)

def demo():
    parser = argparse.ArgumentParser(description='DDPM Demo')
    parser.add_argument('--model', type=str, default='ddpm', choices=['ddpm', 'ddim', 'transformer', 'compressed'],
                        help='Model type to demonstrate')
    parser.add_argument('--steps', type=int, default=50, help='Number of steps for DDIM sampling')
    parser.add_argument('--samples', type=int, default=4, help='Number of samples to generate')
    parser.add_argument('--show_denoising', action='store_true', help='Show the denoising process')
    args = parser.parse_args()

    # Configure the model
    config = Config()
    config.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Using device: {config.device}")

    if args.model == 'ddpm':
        config.model_type = "unet"
        config.sampling_method = "ddpm"
        config.compressed = False
        model_name = "DDPM (U-Net, 1000 steps)"
    elif args.model == 'ddim':
        config.model_type = "unet"
        config.sampling_method = "ddim"
        config.ddim_sampling_steps = args.steps
        config.compressed = False
        model_name = f"DDIM (U-Net, {args.steps} steps)"
    elif args.model == 'transformer':
        config.model_type = "transformer"
        config.sampling_method = "ddpm"
        config.compressed = False
        model_name = "Transformer-based DDPM"
    elif args.model == 'compressed':
        config.model_type = "unet"
        config.sampling_method = "ddpm"
        config.compressed = True
        model_name = "Compressed DDPM"

    # Create model
    model = DiffusionModel(config)

    # Quick minimal training for demo purposes
    print("Running quick training for demo purposes...")
    dataloader = get_data_loader(config)
    model.train(dataloader, num_epochs=1)

    if args.show_denoising:
        visualize_denoising_process(model, args.samples)
    else:
        # Generate and display samples
        print(f"Generating {args.samples} samples with {model_name}...")
        start_time = time.time()
        samples = model.sample(args.samples)
        end_time = time.time()

        # Display the samples
        samples = (samples + 1) * 0.5  # [-1, 1] -> [0, 1]
        samples = samples.clamp(0, 1)
        samples = samples.cpu().permute(0, 2, 3, 1).numpy()

        # Create figure
        plt.figure(figsize=(args.samples * 3, 4))
        for i, sample in enumerate(samples):
            plt.subplot(1, args.samples, i + 1)
            plt.imshow(sample)
            plt.axis('off')

        plt.suptitle(f"{model_name} - Generated Samples\nInference time: {end_time - start_time:.2f}s", y=0.95)
        plt.tight_layout()
        plt.savefig(f'{args.model}_samples.png')
        plt.show()

def visualize_denoising_process(model, num_samples=4):
    """Visualize the step-by-step denoising process"""
    config = model.config
    device = config.device

    # Start with random noise
    x = torch.randn(num_samples, config.channels, config.image_size, config.image_size).to(device)

    # Select visualization steps
    if config.sampling_method == "ddpm":
        steps = 10  # Show 10 steps of the process
        indices = torch.linspace(0, config.time_steps - 1, steps).long().flip(0)
    else:  # DDIM
        steps = min(10, config.ddim_sampling_steps)
        indices = torch.linspace(0, config.ddim_sampling_steps - 1, steps).long().flip(0)

    # Create figure
    plt.figure(figsize=(steps*1.5, num_samples*1.5))

    # Noise image at t=T
    images = [x.clone()]
    visualize_step = 0

    # Denoise step by step
    if config.sampling_method == "ddpm":
        for t in tqdm(range(config.time_steps-1, -1, -1), desc="Denoising"):
            # Skip steps not in visualization list
            if t not in indices:
                continue

            # Sample from the model
            t_batch = torch.full((num_samples,), t, device=device, dtype=torch.long)
            with torch.no_grad():
                predicted_noise = model.model(x, t_batch)

            # Apply the update step
            alpha_t = model.diffusion_params["alphas"][t]
            alpha_cumprod_t = model.diffusion_params["alphas_cumprod"][t]
            beta_t = model.diffusion_params["betas"][t]

            if t > 0:
                noise = torch.randn_like(x)
            else:
                noise = torch.zeros_like(x)

            # Get mean for posterior
            sqrt_recip_alpha_t = 1 / torch.sqrt(alpha_t)
            posterior_mean = sqrt_recip_alpha_t * (
                x - beta_t / torch.sqrt(1 - alpha_cumprod_t) * predicted_noise
            )

            # Add noise scaled by posterior variance
            posterior_var = model.diffusion_params["posterior_variance"][t]
            x = posterior_mean + torch.sqrt(posterior_var) * noise

            # Save for visualization
            images.append(x.clone())
            visualize_step += 1
    else:  # DDIM
        time_steps = torch.linspace(config.time_steps-1, 0, config.ddim_sampling_steps).long().to(device)
        for i, t in enumerate(tqdm(time_steps, desc="Denoising")):
            # Skip steps not in visualization list
            if i not in indices:
                continue

            # Apply DDIM step
            t_batch = t.repeat(num_samples)
            with torch.no_grad():
                predicted_noise = model.model(x, t_batch)

            alpha_cumprod_t = model.diffusion_params["alphas_cumprod"][t]

            if i < len(time_steps) - 1:
                next_t = time_steps[i+1]
                alpha_cumprod_next = model.diffusion_params["alphas_cumprod"][next_t]

                # DDIM update
                x_0 = (x - torch.sqrt(1-alpha_cumprod_t) * predicted_noise) / torch.sqrt(alpha_cumprod_t)
                x_0 = torch.clamp(x_0, -1, 1)
                x = torch.sqrt(alpha_cumprod_next) * x_0 + torch.sqrt(1-alpha_cumprod_next) * predicted_noise
            else:
                # Last step
                x = (x - torch.sqrt(1-alpha_cumprod_t) * predicted_noise) / torch.sqrt(alpha_cumprod_t)

            # Save for visualization
            images.append(x.clone())
            visualize_step += 1

    # Display all steps for all samples
    for i, img_batch in enumerate(images):
        for j in range(num_samples):
            # Convert to displayable format
            sample = (img_batch[j] + 1) * 0.5  # [-1, 1] -> [0, 1]
            sample = sample.clamp(0, 1).cpu().permute(1, 2, 0).numpy()

            # Add to plot
            plt.subplot(num_samples, len(images), i + 1 + j * len(images))
            plt.imshow(sample)
            plt.axis('off')

            if j == 0:
                if i == 0:
                    plt.title("Initial noise")
                else:
                    plt.title(f"Step {i}")

    plt.suptitle(f"Denoising Process - {model.config.model_type.upper()}", y=0.98)
    plt.tight_layout()
    plt.subplots_adjust(wspace=0.1, hspace=0.1)
    plt.savefig('denoising_process.png', dpi=300, bbox_inches='tight')
    plt.show()

def compare_speed():
    """Compare and visualize inference speeds"""
    config = Config()
    config.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("Comparing inference speeds...")

    # Set up models
    models = {
        'DDPM (1000 steps)': {
            'type': 'unet',
            'sampling': 'ddpm',
            'compressed': False,
            'steps': 1000
        },
        'DDIM (50 steps)': {
            'type': 'unet',
            'sampling': 'ddim',
            'compressed': False,
            'steps': 50
        },
        'DDIM (20 steps)': {
            'type': 'unet',
            'sampling': 'ddim',
            'compressed': False,
            'steps': 20
        },
        'Transformer': {
            'type': 'transformer',
            'sampling': 'ddpm',
            'compressed': False,
            'steps': 1000
        },
        'Compressed': {
            'type': 'unet',
            'sampling': 'ddpm',
            'compressed': True,
            'steps': 1000
        }
    }

    # Measure inference times
    times = {}
    for name, params in models.items():
        config.model_type = params['type']
        config.sampling_method = params['sampling']
        config.compressed = params['compressed']
        if params['sampling'] == 'ddim':
            config.ddim_sampling_steps = params['steps']

        print(f"Testing {name}...")
        model = DiffusionModel(config)

        # Quick train to initialize
        dataloader = get_data_loader(config)
        model.train(dataloader, num_epochs=1)

        # Measure inference time
        start_time = time.time()
        _ = model.sample(4)
        times[name] = time.time() - start_time
        print(f"  Time: {times[name]:.2f}s")

    # Calculate speedups
    baseline_time = times['DDPM (1000 steps)']
    speedups = {name: baseline_time / time for name, time in times.items()}

    # Plot results
    plt.figure(figsize=(12, 6))

    names = list(times.keys())
    time_values = [times[name] for name in names]
    speedup_values = [speedups[name] for name in names]

    # Use different colors for each bar
    colors = ['blue', 'green', 'lightgreen', 'orange', 'red']
    bars = plt.bar(names, time_values, color=colors)

    plt.xlabel('Model', fontsize=12)
    plt.ylabel('Inference Time (seconds)', fontsize=12)
    plt.title('Diffusion Model Inference Speed Comparison', fontsize=14)

    # Add time and speedup annotations
    for i, (time_val, speedup) in enumerate(zip(time_values, speedup_values)):
        plt.text(i, time_val + 0.1, f"{time_val:.2f}s\n({speedup:.1f}x faster)",
                 ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()
    plt.savefig('speed_comparison.png', dpi=300)
    plt.show()

    # Print summary table
    print("\nSummary of Results:")
    print(f"{'Model':<20}{'Time (s)':<10}{'Speedup':<10}")
    print("-" * 40)
    for name in names:
        print(f"{name:<20}{times[name]:<10.2f}{speedups[name]:<10.1f}x")

    return times, speedups

def compare_quality():
    """Generate and compare quality of different models"""
    config = Config()
    config.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("Comparing sample quality...")

    # Set up models
    models = {
        'DDPM': {
            'type': 'unet',
            'sampling': 'ddpm',
            'compressed': False
        },
        'DDIM': {
            'type': 'unet',
            'sampling': 'ddim',
            'compressed': False,
            'steps': 50
        },
        'Transformer': {
            'type': 'transformer',
            'sampling': 'ddpm',
            'compressed': False
        },
        'Compressed': {
            'type': 'unet',
            'sampling': 'ddpm',
            'compressed': True
        }
    }

    # Get dataloader (for training)
    dataloader = get_data_loader(config)

    # Generate samples for each model
    samples = {}
    for name, params in models.items():
        config.model_type = params['type']
        config.sampling_method = params['sampling']
        config.compressed = params['compressed']
        if params['sampling'] == 'ddim':
            config.ddim_sampling_steps = params['steps']

        print(f"Generating samples with {name}...")
        model = DiffusionModel(config)

        # Quick train
        model.train(dataloader, num_epochs=1)

        # Generate samples
        samples[name] = model.sample(4)

    # Display samples side by side
    fig, axes = plt.subplots(len(models), 4, figsize=(12, 3*len(models)))

    for i, (name, sample_batch) in enumerate(samples.items()):
        # Convert to displayable format
        sample_batch = (sample_batch + 1) * 0.5  # [-1, 1] -> [0, 1]
        sample_batch = sample_batch.clamp(0, 1).cpu().permute(0, 2, 3, 1).numpy()

        for j in range(4):
            axes[i, j].imshow(sample_batch[j])
            axes[i, j].axis('off')

            if j == 0:
                axes[i, j].set_title(name, fontsize=12, loc='left')

    plt.suptitle("Quality Comparison Across Models", fontsize=16)
    plt.tight_layout()
    plt.subplots_adjust(top=0.93)
    plt.savefig('quality_comparison.png', dpi=300)
    plt.show()

if __name__ == "__main__":
    import sys

    if len(sys.argv) > 1:
        if sys.argv[1] == "speed":
            compare_speed()
        elif sys.argv[1] == "quality":
            compare_quality()
        else:
            demo()
    else:
        demo()

Writing ddpm_demo.py


In [5]:
%%writefile run_demo.py
import subprocess
import time

print("====== DDPM Demo for Video Recording ======")

# 1. Show denoising process
print("\n1. Visualizing the denoising process...")
subprocess.run(["python", "ddpm_demo.py", "--show_denoising"])
time.sleep(3)

# 2. Generate samples with each model
print("\n2. Generating samples with different models...")
models = ["ddpm", "ddim", "transformer", "compressed"]
for model in models:
    print(f"\nGenerating {model} samples...")
    if model == "ddim":
        subprocess.run(["python", "ddpm_demo.py", "--model", model, "--steps", "50"])
    else:
        subprocess.run(["python", "ddpm_demo.py", "--model", model])
    time.sleep(2)

# 3. Compare speeds
print("\n3. Comparing inference speeds...")
subprocess.run(["python", "ddpm_demo.py", "speed"])
time.sleep(3)

# 4. Compare quality
print("\n4. Comparing sample quality...")
subprocess.run(["python", "ddpm_demo.py", "quality"])

print("\nDemo complete! All visualizations saved to files.")

Writing run_demo.py


In [ ]:
!python run_demo.py

====== DDPM Demo for Video Recording ======

1. Visualizing the denoising process...
Using device: cuda
Running quick training for demo purposes...
100% 170M/170M [00:13<00:00, 13.0MB/s]
Epoch 1/1: 100% 782/782 [00:42<00:00, 18.48it/s, loss=0.144]
Epoch 1/1, Average Loss: 0.3060
Sampling: 100% 1000/1000 [00:02<00:00, 450.85it/s]
Denoising: 100% 1000/1000 [00:00<00:00, 31999.76it/s]
Figure(1500x600)

2. Generating samples with different models...

Generating ddpm samples...
Using device: cuda
Running quick training for demo purposes...
Epoch 1/1: 100% 782/782 [00:42<00:00, 18.52it/s, loss=0.139]
Epoch 1/1, Average Loss: 0.3080
Sampling: 100% 1000/1000 [00:02<00:00, 369.54it/s]
Generating 4 samples with DDPM (U-Net, 1000 steps)...
Sampling: 100% 1000/1000 [00:02<00:00, 467.44it/s]
Figure(1200x400)

Generating ddim samples...
Using device: cuda
Running quick training for demo purposes...
Epoch 1/1: 100% 782/782 [00:42<00:00, 18.54it/s, loss=0.155]
Epoch 1/1, Average Loss: 0.3062
DDIM Samp